# Task 2: Comparing Models with Feature Selection and Feature Reduction

This notebook compares a Decision Tree, Gaussian Naive Bayes, and SVM on the Taiwanese Bankruptcy Prediction dataset using all features, the 10 best features selected by mutual information, and 10 PCA components.

In [ ]:
%pip install ucimlrepo scikit-learn pandas

In [ ]:
import pandas as pd
from IPython.display import display, Markdown
from ucimlrepo import fetch_ucirepo

from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

## Load the dataset

In [1]:
data = fetch_ucirepo(id=572)
X = data.data.features.copy()
y = data.data.targets.squeeze().copy()

X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median())
y = pd.to_numeric(y, errors="coerce").astype(int)

print(f"Features: {X.shape[1]}")
print(f"Rows: {X.shape[0]}")
print("Class counts:")
print(y.value_counts().sort_index())

NameError: name 'fetch_ucirepo' is not defined

## Make one shared train/test split

The same 80/20 split is used for every model and every feature representation. Feature selection and PCA are fitted only on the training data to avoid test-set leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

mi_selector = SelectKBest(
    score_func=lambda features, labels: mutual_info_classif(
        features, labels, random_state=42
    ),
    k=10
)
X_train_mi = mi_selector.fit_transform(X_train, y_train)
X_test_mi = mi_selector.transform(X_test)
mi_features = X_train.columns[mi_selector.get_support()].tolist()

print("Top 10 mutual-information features:")
print(mi_features)

pca_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10, random_state=42))
])
X_train_pca = pca_pipeline.fit_transform(X_train, y_train)
X_test_pca = pca_pipeline.transform(X_test)
print(f"PCA variance retained: {pca_pipeline.named_steps['pca'].explained_variance_ratio_.sum():.2%}")

## Train and evaluate the classifiers

In [ ]:
def evaluate_models(setting, X_train_current, X_test_current):
    models = {
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Naive Bayes": GaussianNB(),
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC())
        ])
    }

    setting_results = []
    print(f"\n{'=' * 70}\n{setting}")
    for model_name, model in models.items():
        model.fit(X_train_current, y_train)
        train_predictions = model.predict(X_train_current)
        test_predictions = model.predict(X_test_current)
        train_accuracy = accuracy_score(y_train, train_predictions)
        test_accuracy = accuracy_score(y_test, test_predictions)

        setting_results.append({
            "Setting": setting,
            "Model": model_name,
            "Training accuracy": train_accuracy,
            "Test accuracy": test_accuracy
        })

        print(f"\n{model_name}")
        print(f"Training accuracy: {train_accuracy:.4f}")
        print(f"Test accuracy:     {test_accuracy:.4f}")
        print("Classification report (test):")
        print(classification_report(y_test, test_predictions, zero_division=0))

        reports[(setting, model_name)] = classification_report(
            y_test, test_predictions, output_dict=True, zero_division=0
        )
    return setting_results

reports = {}
results = []
results.extend(evaluate_models("Baseline - all 95 features", X_train, X_test))
results.extend(evaluate_models("Mutual Information - top 10 features", X_train_mi, X_test_mi))
results.extend(evaluate_models("PCA - 10 components", X_train_pca, X_test_pca))

## Accuracy summary

In [ ]:
results_df = pd.DataFrame(results)
display(results_df.style.format({
    "Training accuracy": "{:.4f}",
    "Test accuracy": "{:.4f}"
}))

## Analysis

In [ ]:
best_row = results_df.loc[results_df["Test accuracy"].idxmax()]

def accuracy_for(model_name, setting_text):
    row = results_df[(results_df["Model"] == model_name) & results_df["Setting"].str.startswith(setting_text)].iloc[0]
    return row["Test accuracy"]

tree_values = [accuracy_for("Decision Tree", label) for label in ["Baseline", "Mutual Information", "PCA"]]
nb_values = [accuracy_for("Naive Bayes", label) for label in ["Baseline", "Mutual Information", "PCA"]]
svm_values = [accuracy_for("SVM", label) for label in ["Baseline", "Mutual Information", "PCA"]]

analysis = f"""
**P1.** The best test result in these experiments was **{best_row['Model']} with {best_row['Setting']}**, with a test accuracy of **{best_row['Test accuracy']:.4f}**. The table shows how each classifier changes when the original 95 features are replaced by either the 10 mutual-information features or the 10 PCA components. Mutual information keeps the individual features that are most related to the target, while PCA creates new components that summarize variation across correlated features. Therefore, the better method here is the one with the highest test accuracy in the summary table, while also considering the training-to-test gap.

**P2.** The Decision Tree test accuracies were **{tree_values[0]:.4f}**, **{tree_values[1]:.4f}**, and **{tree_values[2]:.4f}** for the baseline, MI, and PCA setups. Trees can work well with the original variables and do not require scaling, but removing features can remove useful split points; PCA can also make the splits less interpretable. The Naive Bayes test accuracies were **{nb_values[0]:.4f}**, **{nb_values[1]:.4f}**, and **{nb_values[2]:.4f}**. Reducing dimensions can help Naive Bayes by removing noisy variables, although PCA components may be harder to model because they combine many original variables. The SVM test accuracies were **{svm_values[0]:.4f}**, **{svm_values[1]:.4f}**, and **{svm_values[2]:.4f}**. SVM is sensitive to irrelevant features and scale, so either MI or PCA can improve its boundary by making the input smaller and cleaner, but useful discarded information can also reduce accuracy.

**P3.** Correlated features contain overlapping information. Mutual information handles this by selecting a small set of individually informative original features, while PCA combines correlated variables into orthogonal components. This reduces redundancy and can make some models faster or less affected by noise, but it can also remove information that a model uses. The results show the practical trade-off: dimensionality reduction or feature selection is useful when it preserves the predictive signal, and the best choice should be based on the common test-set results rather than training accuracy alone.
"""
display(Markdown(analysis))